In [1]:
import numpy as np
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 100]),
 'token_type_ids': torch.Size([32, 100]),
 'attention_mask': torch.Size([32, 100])}

# 4. Load Model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(1.2996, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [25]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

device

device(type='cuda', index=1)

### Load Trained Weights

In [26]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location=device)
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [27]:
from train_eval_func import eval_loop

In [28]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

# 5. Merge Most Similar Layers

In [29]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sn
from tqdm import tqdm

In [30]:
import os
from merge_helper import merge_bert_layers, LayerMergeTracker
from train_eval_func import train, get_primary_metric, set_lr_scheduler, iterative_cka_merge_and_train, get_recovery_epoch
from torch.optim import AdamW

In [31]:
init_metric = eval_loop(model, val_dataloader, task_name, device)[0]
init_metric

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

In [32]:
tracker = LayerMergeTracker(num_layers=12)

print("Initial state:")
print(f"  {tracker.get_mapping()}")
print(f"  Total layers: {len(tracker)}")
print()

Initial state:
  [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9], [10], [11]]
  Total layers: 12



In [33]:
from CKA import CKAEvaluator

In [34]:
cka_eval = CKAEvaluator(device)

In [35]:
recovery_epochs = get_recovery_epoch(task_name)
recovery_epochs

3

In [36]:
history = iterative_cka_merge_and_train(
    model=model,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    task_name='mrpc',
    device=device,
    cka_evaluator=cka_eval,
    tracker=tracker,
    init_metric=init_metric,
    num_merges=6,
    target_layers=[6, 7, 8],  # Save models with 6, 7, 8 layers
    recovery_epochs=recovery_epochs,
    recovery_lr=1e-5,
    save_dir='./weights/'
)

Target Metric for mrpc: f1
Original f1 score: 0.9094
Recovery threshold: 0.0091

Merge Iteration: 1/6

[1/5] Computing CKA Similarity...


CKA Evaluation: 100%|██████████| 115/115 [00:08<00:00, 14.08it/s]


  Similarity Stats:
    Average: 0.899850
    Highest: 0.993584
    Lowest:  0.723500

[2/5] Merging Layers...
  Merged Layers: 9 + 10
  Layer Composition: [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9, 10], [11]]
  Total Layers: 11

[3/5] Evaluating Post-Merge Performance...
  Metrics:
      accuracy: 0.8627
    ★ f1: 0.9067

[4/5] Recovery Training...
  Performance drop: 0.0027 (threshold: 0.0091)
  → Recovery training NOT needed

[5/5] Saving Model...
  • 11 layers (not in target list, skipping save)

Merge Iteration: 2/6

[1/5] Computing CKA Similarity...


CKA Evaluation: 100%|██████████| 115/115 [00:06<00:00, 16.55it/s]


  Similarity Stats:
    Average: 0.889100
    Highest: 0.983848
    Lowest:  0.723342

[2/5] Merging Layers...
  Merged Layers: 9 + 10
  Layer Composition: [[0], [1], [2], [3], [4], [5], [6], [7], [8], [9, 10, 11]]
  Total Layers: 10

[3/5] Evaluating Post-Merge Performance...
  Metrics:
      accuracy: 0.8848
    ★ f1: 0.9194

[4/5] Recovery Training...
  Performance drop: -0.0100 (threshold: 0.0091)
  → Recovery training NOT needed

[5/5] Saving Model...
  • 10 layers (not in target list, skipping save)

Merge Iteration: 3/6

[1/5] Computing CKA Similarity...


...

✓ Saved checkpoint (best loss: 1.1005)

<----------------- Epoch 1 ----------------->
Loss: 0.06, Training Metrics:
  accuracy: 0.9817
  f1: 0.9865
Validation Loss: 1.1, Validation Metrics:
  accuracy: 0.8456
  f1: 0.8904
Elapsed Time: 13.3256 sec


Training:  67%|██████▋   | 230/345 [00:50<00:11, 10.18it/s]

✓ Saved checkpoint (best loss: 1.1436)

<----------------- Epoch 2 ----------------->
Loss: 0.02, Training Metrics:
  accuracy: 0.9937
  f1: 0.9953
Validation Loss: 1.14, Validation Metrics:
  accuracy: 0.8480
  f1: 0.8916
Elapsed Time: 13.1141 sec


Training: 100%|██████████| 345/345 [01:15<00:00,  4.56it/s]


✓ Saved checkpoint (best loss: 1.2143)

<----------------- Epoch 3 ----------------->
Loss: 0.02, Training Metrics:
  accuracy: 0.9940
  f1: 0.9956
Validation Loss: 1.21, Validation Metrics:
  accuracy: 0.8480
  f1: 0.8938
Elapsed Time: 13.0933 sec

Total Training Time: 75.66 sec
Chosen Primary Metric: f1
Best Metric Run: 0.8938

  → Loading best checkpoint...
  ✓ Loaded best checkpoint from epoch 3
    Best val loss: 1.2143
    Best val metrics: {'accuracy': 0.8480392156862745, 'f1': 0.8938356164383562}
  ✓ Cleaned up temporary checkpoint

[5/5] Saving Model...
  ✓ Saved 6-layer model to ./weights/layer-6-mrpc.pt

MERGE LOOP COMPLETED
Total merges: 6
Final layers: 6
Final composition: [[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]


In [37]:
# See what happened
print("Merge history:")
for i, iteration in enumerate(history['iterations']):
    print(f"  Iteration {iteration['iteration']}:")
    print(f"    Merged: {iteration['merged_layers']}")
    print(f"    Layers after: {iteration['num_layers_after']}")
    print(f"    Composition: {history['layer_compositions'][i]}")
    
    if history['training_stats'][i]:
        final_metrics = history['metrics_after_recovery'][i]
        print(f"    Final metrics: {final_metrics}")

Merge history:
  Iteration 1:
    Merged: (9, 10)
    Layers after: 11
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]
  Iteration 2:
    Merged: (9, 10)
    Layers after: 10
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]
  Iteration 3:
    Merged: (6, 7)
    Layers after: 9
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]
    Final metrics: {'accuracy': 0.8676470588235294, 'f1': 0.90625}
  Iteration 4:
    Merged: (1, 2)
    Layers after: 8
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]
    Final metrics: {'accuracy': 0.8578431372549019, 'f1': 0.8996539792387543}
  Iteration 5:
    Merged: (6, 7)
    Layers after: 7
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]
    Final metrics: {'accuracy': 0.8700980392156863, 'f1': 0.9084628670120898}
  Iteration 6:
    Merged: (4, 5)
    Layers after: 6
    Composition: [[0], [1, 2], [3], [4], [5, 6, 7], [8, 9, 10, 11]]
    Final metrics: {'accur